Procesamiento de Lenguaje Natural  
Proyecto 2 - Chatbot Generador de Tareas para Jira  
Autores: Ing. González Daniel & Ing. Guerra Michael  
Profesor: Dr. Luis Roberto Garcia  

# Credenciales

In [ ]:
import os, base64, requests
from typing import Optional, List, Dict
from getpass import getpass

if "JIRA_BASE" not in os.environ:
    os.environ["JIRA_BASE"] = "https://project.atlassian.net"

os.environ.pop("JIRA_PAT", None)

if "JIRA_EMAIL" not in os.environ:
    os.environ["JIRA_EMAIL"] = input("JIRA_EMAIL: ").strip()
if "JIRA_TOKEN" not in os.environ:
    os.environ["JIRA_TOKEN"] = getpass("JIRA_TOKEN (API token): ").strip()
    
print("✓ Jira configurado con email+token")


✓ Jira configurado con email+token


In [2]:
def _auth_headers():
    email, token = os.getenv("JIRA_EMAIL"), os.getenv("JIRA_TOKEN")
    if not (email and token):
        raise RuntimeError("Faltan JIRA_EMAIL o JIRA_TOKEN")
    b = base64.b64encode(f"{email}:{token}".encode()).decode()
    return {"Authorization": f"Basic {b}"}

def _base():
    base = os.getenv("JIRA_BASE")
    if not base:
        raise RuntimeError("Falta JIRA_BASE")
    return base.rstrip('/')

In [ ]:
class ServidorJira:
    """
    Herramientas MCP-style para Jira:
      - create_issue(projectKey, summary, description?, issueType?, assignee?, labels?, customFields?)
      - create_subtask(parentKey, summary, description?, subtaskTypeName?, subtaskTypeId?, assignee?, labels?, customFields?)
      - search_issues(jql, maxResults?)
      - add_comment(issueKey, comment)
      - list_transitions(issueKey)
      - transition_issue(issueKey, transitionId | transitionName, fields?)
    """
    def __init__(self):
        self.tools = [
            {
                "name": "create_issue",
                "description": "Crea un issue en Jira",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "projectKey": {"type": "string"},
                        "summary": {"type": "string"},
                        "description": {"type": "string"},
                        "issueType": {"type": "string", "default": "Task"},
                        "assignee": {"type": "string", "description": "accountId en Cloud"},
                        "labels": {"type": "array", "items": {"type": "string"}},
                        "customFields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["projectKey", "summary"]
                }
            },
            {
                "name": "create_subtask",
                "description": "Crea una sub-tarea bajo un issue padre",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "parentKey": {"type": "string"},
                        "summary": {"type": "string"},
                        "description": {"type": "string"},
                        "subtaskTypeName": {"type": "string", "default": "Sub-task"},
                        "subtaskTypeId": {"type": "string"},
                        "assignee": {"type": "string"},
                        "labels": {"type": "array", "items": {"type": "string"}},
                        "customFields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["parentKey", "summary"]
                }
            },
            {
                "name": "search_issues",
                "description": "Busca issues por JQL",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "jql": {"type": "string"},
                        "maxResults": {"type": "integer", "default": 10}
                    },
                    "required": ["jql"]
                }
            },
            {
                "name": "add_comment",
                "description": "Agrega un comentario a un issue",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"},
                        "comment": {"type": "string"}
                    },
                    "required": ["issueKey", "comment"]
                }
            },
            {
                "name": "list_transitions",
                "description": "Lista transiciones disponibles para un issue (nombre, id, destino)",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"}
                    },
                    "required": ["issueKey"]
                }
            },
            {
                "name": "transition_issue",
                "description": "Cambia el estado de un issue (por id o nombre de transición)",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "issueKey": {"type": "string"},
                        "transitionId": {"type": "string"},
                        "transitionName": {"type": "string"},
                        "fields": {"type": "object", "additionalProperties": True}
                    },
                    "required": ["issueKey"]
                }
            }
        ]

    def get_tools(self):
        return self.tools

    def ejecutar_herramienta(self, name: str, arguments: dict) -> str:
        try:
            if name == "create_issue":
                return self._create_issue(**arguments)
            if name == "create_subtask":
                return self._create_subtask(**arguments)
            if name == "search_issues":
                return self._search_issues(**arguments)
            if name == "add_comment":
                return self._add_comment(**arguments)
            if name == "list_transitions":
                return self._list_transitions(**arguments)
            if name == "transition_issue":
                return self._transition_issue(**arguments)
            return f"Herramienta '{name}' no encontrada"
        except TypeError as te:
            return f"❌ Parámetros inválidos: {te}"
        except requests.HTTPError as he:
            try:
                return f"❌ HTTP {he.response.status_code}: {he.response.json()}"
            except Exception:
                return f"❌ HTTP {he.response.status_code}: {he.response.text}"
        except Exception as e:
            return f"❌ Error: {e}"

    # ---------------- Implementaciones ----------------
    def _create_issue(self,
                      projectKey: str, summary: str, description: str = "",
                      issueType: str = "Task", assignee: Optional[str] = None,
                      labels: Optional[List[str]] = None,
                      customFields: Optional[Dict[str, object]] = None) -> str:
        url = f"{_base()}/rest/api/3/issue"
        fields = {
            "project": {"key": projectKey},
            "summary": summary,
            "description": description,
            "issuetype": {"name": issueType}
        }
        if assignee:
            fields["assignee"] = {"id": assignee} 
        if labels:
            fields["labels"] = labels
        if customFields:
            fields.update(customFields)

        r = requests.post(url, json={"fields": fields},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        key = r.json().get("key")
        return f"✓ Creado {key} → {_base()}/browse/{key}"

    def _create_subtask(self,
                        parentKey: str, summary: str, description: str = "",
                        subtaskTypeName: str = "Subtask", subtaskTypeId: Optional[str] = None,
                        assignee: Optional[str] = None, labels: Optional[List[str]] = None,
                        customFields: Optional[Dict[str, object]] = None) -> str:
        url = f"{_base()}/rest/api/3/issue"
        issuetype = {"id": subtaskTypeId} if subtaskTypeId else {"name": subtaskTypeName}
        fields = {
            "parent": {"key": parentKey},
            "summary": summary,
            "description": description,
            "issuetype": issuetype
        }
        if assignee:
            fields["assignee"] = {"id": assignee}
        if labels:
            fields["labels"] = labels
        if customFields:
            fields.update(customFields)

        r = requests.post(url, json={"fields": fields},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        key = r.json().get("key")
        return f"✓ Sub-tarea {key} bajo {parentKey} → {_base()}/browse/{key}"

    def _search_issues(self, jql: str, maxResults: int = 10,
                   fields: Optional[List[str]] = None,
                   nextPageToken: Optional[str] = None,
                   expand: Optional[str] = None) -> str:
        base = _base()
        headers = {"Accept": "application/json", **_auth_headers()}

        params = {
            "jql": jql,
            "maxResults": maxResults,
        }
        if fields:
            
            params["fields"] = ",".join(fields)
        if nextPageToken:
            params["nextPageToken"] = nextPageToken
        if expand:
            params["expand"] = expand

        url = f"{base}/rest/api/3/search/jql"
        r = requests.get(url, headers=headers, params=params)

        if r.status_code == 405 or r.status_code == 400:
            body = {
                "jql": jql,
                "maxResults": maxResults,
            }
            if fields:
                body["fields"] = fields
            if nextPageToken:
                body["nextPageToken"] = nextPageToken
            if expand:
                body["expand"] = expand
            r = requests.post(url, headers={"Content-Type":"application/json", **_auth_headers()}, json=body)

        try:
            data = r.json()
        except Exception:
            return f"❌ Error: {r.status_code} - {r.text}"
        if r.status_code >= 400:
            return f"❌ Error {r.status_code}: {data}"

        issues = data.get("issues", [])
        is_last = data.get("isLast", True)
        next_token = data.get("nextPageToken")

        lines = [f"✓ {len(issues)} resultado(s) | isLast={is_last} | nextPageToken={next_token}"]
        for it in issues:
            key = it.get("key")
            flds = it.get("fields", {}) or {}
            summ = flds.get("summary")
            stat = (flds.get("status") or {}).get("name")
            ass  = flds.get("assignee") or {}
            assn = ass.get("displayName") if isinstance(ass, dict) else ass
            labels = flds.get("labels", [])
            lines.append(f"- {key}: {summ} | {stat} | {assn} | labels={labels}")
        return "\n".join(lines)

    def _add_comment(self, issueKey: str, comment: str) -> str:
        url = f"{_base()}/rest/api/3/issue/{issueKey}/comment"
        r = requests.post(url, json={"body": comment},
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        return f"✓ Comentario agregado a {issueKey}"

    def _list_transitions(self, issueKey: str) -> str:
        url = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
        r = requests.get(url, headers={"Accept":"application/json", **_auth_headers()})
        r.raise_for_status()
        trans = r.json().get("transitions", [])
        if not trans:
            return "No hay transiciones disponibles (revisa permisos/flujo)."
        lines = ["Transiciones disponibles:"]
        for t in trans:
            lines.append(f"- {t.get('name')} (id={t.get('id')}) → destino: {(t.get('to') or {}).get('name')}")
        return "\n".join(lines)

    def _transition_issue(self, issueKey: str, transitionId: Optional[str] = None,
                          transitionName: Optional[str] = None,
                          fields: Optional[Dict[str,object]] = None) -> str:
        # Resolver ID por nombre si no se proporciona
        if not transitionId and transitionName:
            url_list = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
            rlist = requests.get(url_list, headers={"Accept":"application/json", **_auth_headers()})
            rlist.raise_for_status()
            opts = rlist.json().get("transitions", [])
            mapping = {t.get("name","").strip().lower(): t.get("id") for t in opts}
            transitionId = mapping.get((transitionName or "").strip().lower())
            if not transitionId:
                nombres = [t.get("name") for t in opts]
                raise ValueError(f"No encontré transición por nombre='{transitionName}'. Disponibles: {nombres}")

        if not transitionId:
            raise ValueError("Debes pasar 'transitionId' o 'transitionName'.")

        payload = {"transition": {"id": str(transitionId)}}
        if fields:
            payload["fields"] = fields  # p.ej. {"resolution": {"name": "Done"}}

        url = f"{_base()}/rest/api/3/issue/{issueKey}/transitions"
        r = requests.post(url, json=payload,
                          headers={"Content-Type":"application/json", **_auth_headers()})
        r.raise_for_status()
        return f"✓ Issue {issueKey} transicionado con transitionId={transitionId}"


In [7]:
def to_adf(text: str) -> dict:
    return {
        "type": "doc",
        "version": 1,
        "content": [
            {"type": "paragraph", "content": [{"type": "text", "text": text}]}
        ]
    }

In [4]:
srv = ServidorJira()
print(srv.get_tools())

[{'name': 'create_issue', 'description': 'Crea un issue en Jira', 'input_schema': {'type': 'object', 'properties': {'projectKey': {'type': 'string'}, 'summary': {'type': 'string'}, 'description': {'type': 'string'}, 'issueType': {'type': 'string', 'default': 'Task'}, 'assignee': {'type': 'string', 'description': 'accountId en Cloud'}, 'labels': {'type': 'array', 'items': {'type': 'string'}}, 'customFields': {'type': 'object', 'additionalProperties': True}}, 'required': ['projectKey', 'summary']}}, {'name': 'create_subtask', 'description': 'Crea una sub-tarea bajo un issue padre', 'input_schema': {'type': 'object', 'properties': {'parentKey': {'type': 'string'}, 'summary': {'type': 'string'}, 'description': {'type': 'string'}, 'subtaskTypeName': {'type': 'string', 'default': 'Sub-task'}, 'subtaskTypeId': {'type': 'string'}, 'assignee': {'type': 'string'}, 'labels': {'type': 'array', 'items': {'type': 'string'}}, 'customFields': {'type': 'object', 'additionalProperties': True}}, 'require

In [5]:
print(srv.ejecutar_herramienta("search_issues", {
    "jql": "issueKey = MCIA-5",
    "fields": ["summary", "status", "assignee", "labels"]
}))


✓ 1 resultado(s) | isLast=True | nextPageToken=None
- MCIA-5: Proyecto PLN | En curso | DANIEL EDUARDO GONZALEZ ALVARADO | labels=[]


In [8]:
print(srv.ejecutar_herramienta("create_subtask",{
    "parentKey": "MCIA-5",
    "subtaskTypeName": "Subtask",
    "summary": "Sub-tarea de prueba",
    "labels": ["subtarea", "prueba"],
    "description": to_adf("Tarea creada desde VS Code"),
    "customFields": {
        "project": {"key": "MCIA"}
    }
}))

✓ Sub-tarea MCIA-7 bajo MCIA-5 → https://dg-mcia.atlassian.net/browse/MCIA-7
